# 04_benchmarking: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/04_benchmarking/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Compute retrieval metrics from their definitions.

Concepts: neighbor recall, precision, reciprocal rank, graded nDCG.
Run: uv run python modules/04_benchmarking/examples/01_from_scratch.py
Expected: recall 0.5, precision 0.5, MRR 0.5
CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

from common.bench.metrics import (
    latency_summary,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
    reciprocal_rank,
)


def main() -> None:
    found = ["flask", "warmer"]
    exact = ["warmer", "chair"]
    judgments = {"warmer": 3, "lamp": 1}
    # KEY IDEA: exact neighbors and useful answers are two different reference sets.
    print("recall:", recall_at_k(found, exact, 2))
    print("precision:", precision_at_k(found, judgments, 2))
    print("MRR:", reciprocal_rank(found, judgments))
    print("nDCG:", round(ndcg_at_k(found, judgments, 2), 4))
    print("latency:", latency_summary([1.0, 2.0, 3.0, 4.0, 100.0]))
    assert (
        recall_at_k(found, exact, 2)
        == precision_at_k(found, judgments, 2)
        == reciprocal_rank(found, judgments)
        == 0.5
    )
    print("Open common/bench/metrics.py: each implementation directly follows its definition.")


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/04_benchmarking/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Evaluate a native LanceDB result against the same judgments.

    Concepts: unindexed exact scan, cosine distance, relevance versus neighbors.
    Run: uv run --extra embedded python modules/04_benchmarking/examples/02_production.py
    Expected: LanceDB result quality and exact neighbor recall recorded
    CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
    Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
    """

    # course-extra: embedded
    import time
    import uuid

    import lancedb

    from common.bench.ground_truth import neighbor_reference
    from common.bench.metrics import ndcg_at_k, precision_at_k, reciprocal_rank, tie_aware_recall
    from common.data import search_queries
    from common.pretty import show_hits
    from common.settings import cache_dir
    from common.teaching import catalog, evidence, ranked_hits

    def main() -> None:
        products, vectors, query, _ = catalog(1000)
        path = cache_dir() / "tier1" / ("lance-" + uuid.uuid4().hex)
        db = lancedb.connect(str(path))
        try:
            table = db.create_table(
                "products",
                data=[
                    {"id": key, "vector": v.tolist()}
                    for key, v in zip(products["id"], vectors, strict=True)
                ],
            )
            start = time.perf_counter()
            rows = table.search(query).distance_type("cosine").limit(10).to_list()
            elapsed = (time.perf_counter() - start) * 1000
            ids = [row["id"] for row in rows]
            truth = neighbor_reference(
                vectors, products["id"].tolist(), products.to_dict("records"), query, 10, "cosine"
            )
            judgments = {
                key: grade
                for key, grade in search_queries()[0]["qrels"].items()
                if key in set(products["id"])
            }
            metrics = {
                "recall": tie_aware_recall(ids, truth, 10),
                "precision": precision_at_k(ids, judgments, 10),
                "mrr": reciprocal_rank(ids, judgments),
                "ndcg": ndcg_at_k(ids, judgments, 10),
                "retrieval_ms": elapsed,
            }
            # KEY IDEA: no ANN index was built; poor relevance cannot be blamed on approximate retrieval here.
            show_hits(
                ranked_hits(products, [(r["id"], 1 - r["_distance"]) for r in rows]),
                elapsed,
                title="LanceDB judged query",
            )
            print(metrics)
            evidence("lancedb_quality", metrics)
        finally:
            db.close()

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/04_benchmarking/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Run a repeatable TinyVec benchmark from a strict configuration.

Concepts: warmup, repetitions, provenance, quality and timing separation.
Run: uv run python modules/04_benchmarking/examples/03_measure.py
Expected: 100 queries x 5 repeats; saved raw run and summary
CPU: about 10–30 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

from common.bench.config import load_config
from common.bench.runner import run_benchmark
from common.settings import ROOT, cache_dir


def main() -> None:
    config = load_config(ROOT / "configs/bench/benchmark_method.yaml")
    # KEY IDEA: loading and embedding are outside the retrieval timer; the raw report states the scope.
    result = run_benchmark(config, cache_dir() / "tier1" / "example-benchmarks")
    print(result["run_id"], result["summary"])
    assert result["summary"]["samples"] == result["query_count"] * config.repetitions
    assert result["summary"]["recall_at_k"] == 1.0


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.bench.metrics import precision_at_k, reciprocal_rank

# Move the relevant mug through the ranking.
found = ["chair", "mug", "lamp"]
print(precision_at_k(found, {"mug": 3}, 3), reciprocal_rank(found, {"mug": 3}))

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.